# Section 0. Setup

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.utils import class_weight
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
import whisper
from transformers import pipeline
import torch
from collections import defaultdict
from datetime import datetime

# Reproducibility for Reviewer Validation
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

class Config:
    # Data Settings
    DATA_PATH = 'dataset_meld/'
    FILE_PREFIX = 'prepared_'
    SR = 16000
    MAX_DURATION = 3
    MAX_SAMPLES = SR * MAX_DURATION
    
    # Feature Settings
    N_MFCC = 40
    MAX_MFCC_LEN = 94  
    
    # Training Parameters 
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 30
    DROPOUT = 0.3
    
    DEFAULT_CLASSES = ['neutral', 'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise']

print("=== Hyperparameters ===")
for k, v in vars(Config).items():
    if not k.startswith('__'): print(f"{k}: {v}")

In [ ]:
import shutil
import imageio_ffmpeg

# 1. Get the directory where imageio_ffmpeg stores its binary
ffmpeg_bin_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())

# 2. Whisper looks specifically for "ffmpeg.exe". Let's make sure a copy exists with that exact name.
target_exe = os.path.join(ffmpeg_bin_dir, "ffmpeg.exe")
if not os.path.exists(target_exe):
    shutil.copy(imageio_ffmpeg.get_ffmpeg_exe(), target_exe)

# 3. Inject this directory into the system PATH for this session
os.environ["PATH"] += os.pathsep + ffmpeg_bin_dir

# Section 1. ASR & Sentiment Classifier

### test with GPU

In [ ]:
import torch
print(torch.__version__)

In [ ]:
# 1. Check GPU availability and set up device formatting
cuda_available = torch.cuda.is_available()
whisper_device = "cuda" if cuda_available else "cpu"
hf_device = 0 if cuda_available else -1

print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"Activating GPU: {torch.cuda.get_device_name(0)}\n")
else:
    print("GPU not detected. Running on CPU instead.\n")

In [ ]:
# 2. Load Whisper ASR explicitly on the target device
print("Loading Whisper ASR...")
asr_model = whisper.load_model("base", device=whisper_device)

# 3. Load RoBERTa Sentiment Classifier on the target device
print("Loading RoBERTa Sentiment Classifier...")
sentiment_classifier = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    device=hf_device
)

print("\n✅ Both models loaded and pinned to GPU successfully!")

In [ ]:
import jiwer

AUDIO_DIRECTORY = "dataset_meld/audio/"
AUDIO_EXT = ".wav"
splits = ['train', 'dev', 'test']

os.makedirs("dataset_meld", exist_ok=True)

# Create a timestamp string for the filenames (e.g., "20260617_2030")
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# --- GLOBAL STATS (per split) ---
stats = defaultdict(int)
failed_samples = []

# Define the text transformation pipeline for accurate WER comparison (Text Normalisation)
transformation = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.Strip()
])

# --- PROCESSING LOOP ---
for split in splits:
    print(f"\n================ {split.upper()} SPLIT ================\n")

    orig_csv = os.path.join("dataset_meld", f"{split}_sent_emo.csv")
    out_csv = os.path.join("dataset_meld", f"prepared_{split}_{timestamp}_sent_emo.csv")

    print(f"Processing: {orig_csv}")
    df = pd.read_csv(orig_csv)

    # --- MOCK RUN MECHANISM ---
    MOCK_MODE = True  # Toggle this to False to run the full dataset
    if MOCK_MODE:
        # Take 10% of the data, sampled randomly
        df = df.sample(frac=0.10, random_state=SEED) 
        print(f"⚠️ MOCK MODE ACTIVE: Processing only 10% of {split} ({len(df)} samples)")
    # --------------------------

    prepared_data = []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']

        raw_path = os.path.join(
            AUDIO_DIRECTORY,
            f"wav_{split}",
            f"dia{dia_id}_utt{utt_id}{AUDIO_EXT}"
        )
        file_path = os.path.abspath(raw_path)

        # ---------------------------
        # Missing audio
        # ---------------------------
        if not os.path.exists(file_path):
            stats["missing_audio"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "missing_audio"
            })
            continue

        # ---------------------------
        # Validate CSV fields
        # ---------------------------
        if pd.isna(row['Emotion']) or pd.isna(row['Sentiment']) or pd.isna(row['Utterance']):
            stats["invalid_csv_data"] += 1
            failed_samples.append({
                "split": split,
                "file_path": file_path,
                "reason": "invalid_csv_data"
            })
            continue

        emo_gt = str(row['Emotion']).lower()
        sent_gt = str(row['Sentiment']).lower()
        trans_gt = str(row['Utterance']).strip()

        # ---------------------------
        # ASR transcription
        # ---------------------------
        try:
            with torch.no_grad():
                trans_asr = asr_model.transcribe(
                    file_path,
                    fp16=True,
                    language="en",
                    task="transcribe"
                )['text'].strip()

        except Exception as e:
            stats["asr_failed"] += 1
            failed_samples.append({
                "split": split, "file_path": file_path, "reason": "asr_failed", "error": str(e)
            })
            continue

        if not trans_asr:
            stats["empty_transcription"] += 1
            failed_samples.append({
                "split": split, "file_path": file_path, "reason": "empty_transcription"
            })
            continue

        # ---------------------------
        # WER Calculation (Text Normalisation applied)
        # ---------------------------
        wer_score = jiwer.wer(transformation(trans_gt), transformation(trans_asr))

        # ---------------------------
        # Sentiment prediction
        # ---------------------------
        try:
            bert_out = sentiment_classifier(trans_asr)[0]
            sent_pred = bert_out['label'].lower()
            sent_score = bert_out['score']

        except Exception as e:
            stats["sentiment_failed"] += 1
            failed_samples.append({
                "split": split, "file_path": file_path, "reason": "sentiment_failed", "error": str(e)
            })
            continue

        # ---------------------------
        # SUCCESS CASE
        # ---------------------------
        prepared_data.append({
            'emotion': emo_gt,
            'sentiment': sent_gt,  
            'textual_sentiment_predicted': sent_pred,
            'textual_sentiment_score': sent_score,
            'transcription_ground_truth': trans_gt,
            'transcription_ASR': trans_asr,
            'WER': wer_score,
            'file_path': file_path
        })

        stats["success"] += 1

    # ---------------------------
    # Save entire batch at once (Optimised for GPU speed)
    # ---------------------------
    if prepared_data:
        chunk_df = pd.DataFrame(prepared_data)
        chunk_df.to_csv(out_csv, index=False)

    print(f"\n✅ Finished {split}. Saved to {out_csv}")

# ---------------------------
# FINAL REPORT
# ---------------------------
print("\n================ FINAL DATASET REPORT ================\n")

total_failures = (
    stats["missing_audio"] + stats["invalid_csv_data"] +
    stats["asr_failed"] + stats["empty_transcription"] +
    stats["sentiment_failed"]
)

print(f"Success samples           : {stats['success']}")
print(f"Missing audio             : {stats['missing_audio']}")
print(f"Invalid CSV data          : {stats['invalid_csv_data']}")
print(f"ASR failed                : {stats['asr_failed']}")
print(f"Empty transcription       : {stats['empty_transcription']}")
print(f"Sentiment prediction fail : {stats['sentiment_failed']}")
print(f"\nTOTAL FAILURES            : {total_failures}")

# ---------------------------
# SAVE FAILURE LOG
# ---------------------------
failed_df = pd.DataFrame(failed_samples)
log_filename = f"dataset_meld/failed_samples_log_{timestamp}.csv"
failed_df.to_csv(log_filename, index=False)

print(f"\n📁 Failure log saved: {log_filename}")